## 8️⃣ Comparación y Selección de Estrategia

### 📊 Tabla Comparativa

| Estrategia | Ventajas | Desventajas | Cuándo Usar |
|------------|----------|-------------|---------------|
| **K-Fold** | Simple, rápido, general | No para datos especiales | Datos i.i.d. aleatorios |
| **Stratified K-Fold** | Preserva distribución | Solo clasificación/bins | Clases desbalanceadas |
| **Time Series CV** | Respeta temporalidad | Solo expanding window | Series temporales |
| **Group K-Fold** | Evita leakage grupos | Puede desbalancear folds | Datos agrupados (clientes) |
| **LOOCV** | Máxima precisión | Extremadamente lento | Datasets pequeños (<100) |
| **Spatial CV** | Evalúa generalización geográfica | Requiere coordenadas | Datos geoespaciales |

---

### 🧩 Árbol de Decisión: ¿Qué Estrategia Usar?

```
¿Tienes datos temporales (fechas)?
│
├── SÍ → Time Series CV
│
└── NO → ¿Tienes grupos (clientes, usuarios, dispositivos)?
    │
    ├── SÍ → Group K-Fold
    │
    └── NO → ¿Tienes coordenadas geográficas?
        │
        ├── SÍ → Spatial CV (Group K-Fold con H3)
        │
        └── NO → ¿Es clasificación con clases desbalanceadas?
            │
            ├── SÍ → Stratified K-Fold
            │
            └── NO → ¿Dataset muy pequeño (<100 registros)?
                │
                ├── SÍ → 10-Fold CV o LOOCV
                │
                └── NO → K-Fold estándar (5 o 10 folds)
```

---

### 📊 Ejemplo Comparativo: Dataset de Panadería

**Dataset**: 50,000 ventas de 500 clientes en 3 sucursales, con fechas

```python
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, TimeSeriesSplit, GroupKFold
from sklearn.ensemble import RandomForestRegressor

# Cargar datos
df = pd.read_csv('ventas.csv')
X = df[['dia_semana', 'mes', 'sucursal_id']]
y = df['total']

model = RandomForestRegressor(n_estimators=100, random_state=42)

# 1. K-Fold estándar
scores_kfold = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')

# 2. Time Series CV
df_sorted = df.sort_values('fecha')
X_ts = df_sorted[['dia_semana', 'mes', 'sucursal_id']]
y_ts = df_sorted['total']
scores_ts = cross_val_score(model, X_ts, y_ts, cv=TimeSeriesSplit(n_splits=5), scoring='neg_mean_absolute_error')

# 3. Group K-Fold (por cliente)
groups = df['cliente_id']
scores_group = cross_val_score(model, X, y, groups=groups, cv=GroupKFold(n_splits=5), scoring='neg_mean_absolute_error')

# Comparar
print("Comparación de Estrategias de CV:")
print(f"K-Fold estándar:     MAE = ${-scores_kfold.mean():.2f}  (puede ser optimista)")
print(f"Time Series CV:      MAE = ${-scores_ts.mean():.2f}  (realista para forecasting)")
print(f"Group K-Fold:        MAE = ${-scores_group.mean():.2f}  (realista para clientes nuevos)")
```

**Salida esperada**:
```
Comparación de Estrategias de CV:
K-Fold estándar:     MAE = $14.50  (puede ser optimista)
Time Series CV:      MAE = $17.20  (realista para forecasting)
Group K-Fold:        MAE = $18.80  (realista para clientes nuevos)
```

💡 **Insights**:
- K-Fold estándar subestima el error (data leakage)
- Time Series CV es más realista para predicciones futuras
- Group K-Fold es más conservador (clientes nunca vistos)

---

### ⚖️ Trade-offs

| Dimensión | K-Fold | Stratified | Time Series | Group | LOOCV | Spatial |
|-----------|--------|------------|-------------|-------|-------|----------|
| **Velocidad** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐ | ⭐⭐⭐⭐ |
| **Precisión** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Uso de datos** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Generalidad** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |

---

## 9️⃣ Mejores Prácticas de Cross-Validation

### ✅ DO: Buenas Prácticas

#### 1. **Seleccionar Estrategia Según Tipo de Datos**
```python
# ✅ BIEN: Time Series CV para datos temporales
df_sorted = df.sort_values('fecha')
scores = cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5))

# ❌ MAL: K-Fold para datos temporales
scores = cross_val_score(model, X, y, cv=5)  # Data leakage!
```

#### 2. **Siempre Usar `random_state` para Reproducibilidad**
```python
# ✅ BIEN
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ❌ MAL (resultados no reproducibles)
kf = KFold(n_splits=5, shuffle=True)  # Diferente cada vez
```

#### 3. **Reportar Media ± Desviación Estándar**
```python
# ✅ BIEN
scores = cross_val_score(model, X, y, cv=5)
print(f"MAE: {-scores.mean():.2f} ± {scores.std():.2f}")

# ❌ MAL (solo media, sin varianza)
print(f"MAE: {-scores.mean():.2f}")
```

#### 4. **Usar Mismo CV para Comparar Modelos**
```python
# ✅ BIEN: Mismo CV para comparación justa
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_rf = cross_val_score(model_rf, X, y, cv=kf)
scores_gb = cross_val_score(model_gb, X, y, cv=kf)

# ❌ MAL: CVs diferentes
scores_rf = cross_val_score(model_rf, X, y, cv=5)
scores_gb = cross_val_score(model_gb, X, y, cv=3)  # No comparable
```

#### 5. **Pre-procesamiento Dentro del Fold**
```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ✅ BIEN: Scaler dentro del pipeline (se ajusta en cada fold)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor())
])
scores = cross_val_score(pipeline, X, y, cv=5)

# ❌ MAL: Scaler fuera (usa info de test en train)
X_scaled = StandardScaler().fit_transform(X)  # Data leakage!
scores = cross_val_score(model, X_scaled, y, cv=5)
```

---

### ❌ DON'T: Errores Comunes

#### 1. **NO Usar K-Fold para Series Temporales**
```python
# ❌ MAL
df_temporal = pd.read_csv('ventas_diarias.csv')
scores = cross_val_score(model, X, y, cv=5)  # Entrena con futuro!

# ✅ BIEN
df_temporal = df_temporal.sort_values('fecha')
scores = cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5))
```

#### 2. **NO Olvidar `groups=` en Group K-Fold**
```python
# ❌ MAL
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(model, X, y, cv=gkf)  # Error o leakage!

# ✅ BIEN
scores = cross_val_score(model, X, y, groups=df['cliente_id'], cv=gkf)
```

#### 3. **NO Hacer Feature Engineering con Datos de Test**
```python
# ❌ MAL: Crear features usando TODO el dataset
df['mean_by_client'] = df.groupby('cliente_id')['total'].transform('mean')  # Leakage!
scores = cross_val_score(model, X, y, cv=5)

# ✅ BIEN: Crear features dentro del fold (usar Pipeline o FunctionTransformer)
```

#### 4. **NO Usar LOOCV con Datasets Grandes**
```python
# ❌ MAL: 10,000 modelos a entrenar
scores = cross_val_score(model, X, y, cv=LeaveOneOut())  # Muy lento!

# ✅ BIEN: 5 o 10 folds es suficiente
scores = cross_val_score(model, X, y, cv=10)
```

#### 5. **NO Ignorar Advertencias de Clases Faltantes**
```python
# ❌ MAL: Ignorar warning de clases desbalanceadas
scores = cross_val_score(model, X, y, cv=5)  # Algunos folds sin clase minoritaria

# ✅ BIEN: Usar Stratified K-Fold
scores = cross_val_score(model, X, y, cv=StratifiedKFold(n_splits=5))
```

---

### 📊 Cuántos Folds Usar?

**Recomendaciones generales**:

| Tamaño Dataset | Nº Folds | Razón |
|-----------------|-----------|--------|
| < 100 | 10 o LOOCV | Maximizar datos de entrenamiento |
| 100 - 1,000 | 10 | Buen balance |
| 1,000 - 10,000 | 5 | Velocidad vs. precisión |
| > 10,000 | 3-5 | Suficiente para estimación robusta |

---

### ⚙️ Parámetros Importantes

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    estimator=model,           # Modelo a evaluar
    X=X,                       # Features
    y=y,                       # Target
    groups=groups,             # Grupos (para GroupKFold)
    cv=5,                      # Estrategia CV (int o objeto)
    scoring='neg_mean_absolute_error',  # Métrica
    n_jobs=-1,                 # Paralelización (-1 = todos los cores)
    verbose=1,                 # Mostrar progreso
    fit_params=None,           # Parámetros extras para fit()
    pre_dispatch='2*n_jobs'    # Control de memoria
)
```

**Métricas comunes**:
- Regresión: `'neg_mean_absolute_error'`, `'neg_mean_squared_error'`, `'r2'`
- Clasificación: `'accuracy'`, `'f1'`, `'roc_auc'`, `'precision'`, `'recall'`

---

### 💡 Tips Finales

1. ✅ **Siempre visualiza la distribución** de train/test en cada fold
2. ✅ **Verifica que no haya data leakage** (revisa features y timestamps)
3. ✅ **Usa nested CV** para optimización de hiperparámetros + evaluación
4. ✅ **Documenta la estrategia CV** usada en tus experimentos
5. ✅ **Considera el costo computacional** (LOOCV vs. 5-Fold)
6. ✅ **Reporta intervalos de confianza** (media ± std)

---

In [0]:
# Instalar dependencias
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    cross_val_score, cross_validate,
    KFold, StratifiedKFold, TimeSeriesSplit, GroupKFold, LeaveOneOut
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score

print("✅ Librerías importadas")

In [0]:
# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')

print("✅ Datasets cargados")
print(f"   Ventas: {len(df_ventas):,}")
print(f"   Clientes: {len(df_clientes):,}")

# Preparar datos
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas = df_ventas.sort_values('fecha').reset_index(drop=True)

df_ventas['dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['dia_mes'] = df_ventas['fecha'].dt.day

print("\n✅ Features temporales creadas")

In [0]:
print("="*80)
print("COMPARACIÓN DE ESTRATEGIAS DE CROSS-VALIDATION")
print("="*80)

# Preparar datos
df_ml = df_ventas[df_ventas['cliente_id'].notna()].merge(
    df_clientes[['cliente_id', 'segmento']], 
    on='cliente_id'
)
df_ml['segmento_encoded'] = df_ml['segmento'].astype('category').cat.codes

features = ['sucursal_id', 'dia_semana', 'mes', 'segmento_encoded']
X = df_ml[features]
y = df_ml['total']

model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)

print(f"\n📊 Dataset: {len(X):,} registros, {len(features)} features")
print(f"\nProbando diferentes estrategias de CV...\n")

# 1. K-Fold estándar
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)

# 2. Time Series CV
scores_ts = cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5), scoring='neg_mean_absolute_error', n_jobs=-1)

# 3. Group K-Fold (por cliente)
groups = df_ml['cliente_id'].values
gkf = GroupKFold(n_splits=5)
scores_gkf = cross_val_score(model, X, y, groups=groups, cv=gkf, scoring='neg_mean_absolute_error', n_jobs=-1)

# Resultados
print("Resultados:")
print(f"\n1️⃣ K-Fold estándar (5 folds):")
print(f"   MAE: ${-scores_kf.mean():.2f} ± ${scores_kf.std():.2f}")
print(f"   💡 Puede ser optimista (no considera estructura de datos)")

print(f"\n2️⃣ Time Series CV (5 folds):")
print(f"   MAE: ${-scores_ts.mean():.2f} ± ${scores_ts.std():.2f}")
print(f"   💡 Realista para forecasting (respeta orden temporal)")

print(f"\n3️⃣ Group K-Fold (por cliente):")
print(f"   MAE: ${-scores_gkf.mean():.2f} ± ${scores_gkf.std():.2f}")
print(f"   💡 Realista para clientes nuevos (evita data leakage)")

print(f"\n" + "="*80)
print("CONCLUSIÓN")
print("="*80)
diff_ts = (-scores_ts.mean()) - (-scores_kf.mean())
diff_gkf = (-scores_gkf.mean()) - (-scores_kf.mean())

print(f"\nDiferencia Time Series vs. K-Fold: ${diff_ts:.2f} ({(diff_ts/-scores_kf.mean()*100):.1f}% peor)")
print(f"Diferencia Group K-Fold vs. K-Fold: ${diff_gkf:.2f} ({(diff_gkf/-scores_kf.mean()*100):.1f}% peor)")
print(f"\n💡 La estrategia correcta revela el VERDADERO rendimiento del modelo.")

## ✅ Conclusiones

### 🎯 Resumen del Módulo

**Lo que aprendimos**:

1. ✅ **Limitaciones de train/test split simple** y por qué usar CV
2. ✅ **K-Fold** para datos aleatorios (i.i.d.)
3. ✅ **Stratified K-Fold** para clases desbalanceadas
4. ✅ **Time Series CV** para respetar dependencia temporal
5. ✅ **Group K-Fold** para evitar data leakage por agrupamiento
6. ✅ **LOOCV** para datasets muy pequeños
7. ✅ **Validación espacial** con features H3
8. ✅ **Cómo seleccionar** la estrategia correcta

---

### 💡 Mensajes Clave

1. 🔑 **No hay una estrategia "mejor"** - depende de tus datos
2. ⚠️ **La estrategia incorrecta** puede dar resultados demasiado optimistas
3. 🎯 **Siempre considera la estructura** de tus datos (temporal, grupos, espacial)
4. ⏱️ **Balance entre precisión y velocidad**: 5-Fold es generalmente suficiente
5. 🔄 **Reproducibilidad**: Siempre usar `random_state`

---

### 📦 Guía Rápida de Selección
**¿Qué CV usar?**

- 📅 **Datos con fechas** → Time Series CV
- 👥 **Múltiples registros por entidad** → Group K-Fold
- 🗺️ **Datos geoespaciales** → Spatial CV (Group K-Fold con H3)
- ⚖️ **Clases desbalanceadas** → Stratified K-Fold
- 🔬 **Dataset < 100 registros** → 10-Fold o LOOCV
- 🎲 **Datos aleatorios sin estructura especial** → K-Fold (5 o 10 folds)

---

### 📚 Recursos Adicionales

- [Scikit-learn Cross-Validation](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Time Series Cross-Validation Best Practices](https://towardsdatascience.com/time-series-cross-validation)
- [Spatial Cross-Validation in R and Python](https://geocompr.robinlovelace.net/)

---

## 🎓 ¡Felicitaciones!

**Has completado el módulo de Validación Cruzada Avanzada.**

Ahora puedes:
- ✅ Seleccionar la estrategia de CV correcta según tipo de datos
- ✅ Evitar data leakage en evaluación de modelos
- ✅ Obtener estimaciones realistas de rendimiento
- ✅ Aplicar CV espacial con features H3
- ✅ Comparar modelos de forma justa

**Próximo paso**: Notebook Práctico con ejercicios hands-on.

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas)**  
**Mendoza, Argentina**

## 5️⃣ Group K-Fold (Evitar Data Leakage)

### 🚨 Problema: Data Leakage por Agrupamiento

**Escenario**: Dataset con **múltiples registros del mismo cliente/usuario/entidad**.

```
Dataset de ventas de panadería:
├── Cliente A: 50 compras
├── Cliente B: 30 compras
├── Cliente C: 20 compras
└── ...
```

❌ **Con K-Fold estándar**:
```
Train: [Cliente A - compra 1, Cliente A - compra 2, ...]
Test:  [Cliente A - compra 45, Cliente A - compra 46, ...]
```

**Problema**: El modelo **aprende sobre Cliente A** en training y lo **evalúa en test** → **data leakage**

✅ **Rendimiento inflado artificialmente** porque el modelo ya "conoce" al cliente.

---

### ✅ Solución: Group K-Fold

**Concepto**: Asegurar que **todos los registros de un grupo estén en el mismo fold**.

```
Fold 1:
  Train: [Cliente B, Cliente C, Cliente D, ...]
  Test:  [Cliente A - todas sus compras]
  
Fold 2:
  Train: [Cliente A, Cliente C, Cliente D, ...]
  Test:  [Cliente B - todas sus compras]
```

✅ **Ventaja**: Evalúa la capacidad del modelo de **generalizar a nuevos grupos** (clientes nunca vistos).

---
### 💻 Implementación

```python
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Definir grupos (IDs de clientes)
groups = df['cliente_id'].values

# Group K-Fold
gkf = GroupKFold(n_splits=5)

model = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation con grupos
scores = cross_val_score(
    model, X, y, 
    groups=groups,  # 🔑 Importante: pasar grupos
    cv=gkf,
    scoring='neg_mean_absolute_error'
)

print(f"MAE: {-scores.mean():.2f} ± {scores.std():.2f}")
```

---

### 📊 Ejemplo Visual

**Dataset**: 1000 compras de 100 clientes

**K-Fold estándar** (INCORRECTO):
```
Train:  [C1-compra1, C1-compra2, C2-compra1, C3-compra1, ...]
Test:   [C1-compra3, C2-compra2, C3-compra2, ...]  ❌ Leakage!
```

**Group K-Fold** (CORRECTO):
```
Train:  [C1-todas, C2-todas, C3-todas, ..., C80-todas]
Test:   [C81-todas, C82-todas, ..., C100-todas]     ✅ Sin leakage
```

---

### 🎯 Casos de Uso

✅ **Usar Group K-Fold cuando**:

1. **Múltiples transacciones por cliente**:
   - Predicción de churn
   - Recomendaciones
   - Credit scoring

2. **Múltiples mediciones por sujeto**:
   - Datos médicos (pacientes)
   - Experimentos (participantes)
   - Sensores (dispositivos)

3. **Datos jerárquicos**:
   - Estudiantes en escuelas
   - Empleados en empresas
   - Productos en categorías

4. **Datos geoespaciales agrupados**:
   - Ventas por sucursal
   - Mediciones por región

---

### ⚠️ Consideraciones

**1. Distribución desigual de grupos**:
```python
# Algunos clientes tienen 1 compra, otros 100
# Los folds pueden ser desbalanceados en tamaño
```

**Solución**: Verificar distribución antes de CV:
```python
print(df.groupby('cliente_id').size().describe())
```

**2. Grupos muy pequeños o muy grandes**:
- Grupo con 1 registro → poca información
- Grupo con 1000 registros → domina el fold

**Solución**: Filtrar o balancear grupos si es necesario.

---

### 🔄 Comparación: K-Fold vs. Group K-Fold

**Experimento**: Predicción de ventas por cliente

| Método | MAE Test | Comentario |
|--------|----------|------------|
| K-Fold estándar | $12.50 | ❌ **Optimista** (data leakage) |
| Group K-Fold | $18.30 | ✅ **Realista** (clientes nuevos) |

💡 **Diferencia de $5.80**: El impacto del data leakage!

---

### 💻 Ejemplo Completo: Panadería

```python
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Cargar datos
df_ventas = pd.read_csv('ventas.csv')

# Features y target
X = df_ventas[['dia_semana', 'mes', 'sucursal_id', 'segmento_encoded']]
y = df_ventas['total']
groups = df_ventas['cliente_id']  # Agrupar por cliente

# Group K-Fold
gkf = GroupKFold(n_splits=5)
model = RandomForestRegressor(n_estimators=100, random_state=42)

scores = cross_val_score(model, X, y, groups=groups, cv=gkf, scoring='neg_mean_absolute_error')

print(f"MAE promedio: ${-scores.mean():.2f}")
print(f"Desviación estándar: ${scores.std():.2f}")
print(f"\n💡 Este MAE es realista para clientes NUEVOS")
```
---

## 6️⃣ Leave-One-Out Cross-Validation (LOOCV)

### 📖 Concepto

**LOOCV** es el caso extremo de K-Fold donde **K = N** (número de registros).

**Proceso**:
- Para cada registro i:
  - Train: Todos los demás registros (N-1)
  - Test: Solo el registro i
- Entrenar **N modelos**

**Visualización** (Dataset con 5 registros):
```
Fold 1: [TEST] [TRAIN] [TRAIN] [TRAIN] [TRAIN]
Fold 2: [TRAIN] [TEST] [TRAIN] [TRAIN] [TRAIN]
Fold 3: [TRAIN] [TRAIN] [TEST] [TRAIN] [TRAIN]
Fold 4: [TRAIN] [TRAIN] [TRAIN] [TEST] [TRAIN]
Fold 5: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]
```

---

### ⚖️ Ventajas y Desventajas

✅ **Ventajas**:
1. **Usa casi todos los datos** para entrenamiento (N-1)
2. **Estimación casi insesgada** del error
3. **Determinístico**: no depende de random splits
4. **Útil con datasets pequeños** (< 100 registros)

❌ **Desventajas**:
1. **Extremadamente lento**: N entrenamientos
   - 1000 registros = 1000 modelos a entrenar
2. **Alta varianza** en la estimación
3. **No funciona con algoritmos estocásticos** (redes neuronales)
4. **Desperdicio computacional** con datasets grandes

---

### 💻 Implementación

```python
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import Ridge

# LOOCV
loo = LeaveOneOut()

# Usar modelo rápido (regresión lineal)
model = Ridge(alpha=1.0)

# Cross-validation (puede ser MUY lento)
scores = cross_val_score(
    model, X, y, 
    cv=loo,
    scoring='neg_mean_squared_error'
)

print(f"Número de folds: {len(scores)}  # = N registros")
print(f"MSE: {-scores.mean():.2f} ± {scores.std():.2f}")
```

---

### ⏱️ Comparación de Tiempo de Ejecución

**Dataset**: 1000 registros, Random Forest con 100 árboles

| Método | Nº Modelos | Tiempo Estimado |
|---------|-------------|------------------|
| Train/Test Split | 1 | 1 segundo |
| 5-Fold CV | 5 | 5 segundos |
| 10-Fold CV | 10 | 10 segundos |
| **LOOCV** | **1000** | **1000 segundos (16 min)** |

🐢 **LOOCV es 200x más lento que 5-Fold!**

---

### 🎯 Cuándo Usar LOOCV

✅ **Usar cuando**:
- Dataset **muy pequeño** (N < 100)
- Modelo **rápido de entrenar** (regresión lineal, KNN)
- Necesitas **máxima precisión** en estimación
- No tienes restricciones de tiempo

❌ **NO usar cuando**:
- Dataset **grande** (N > 500)
- Modelo **lento** (RF, GB, redes neuronales)
- Tienes **tiempo limitado**
- **Casi siempre**: 5-Fold o 10-Fold es mejor opción

---

### 💡 Alternativa: Leave-P-Out

**Leave-P-Out CV**: Dejar **P registros** fuera en cada fold.

```python
from sklearn.model_selection import LeavePOut

# Dejar 2 registros fuera
lpo = LeavePOut(p=2)

# Número de combinaciones = C(N, 2)
# Para N=100: 4,950 combinaciones!
```

⚠️ **Aún más lento que LOOCV**. Raramente usado en práctica.

---

### 📊 Ejemplo Práctico: Dataset Pequeño

```python
import numpy as np
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression

# Dataset pequeño (50 registros)
np.random.seed(42)
X = np.random.randn(50, 5)
y = np.random.randn(50)

# LOOCV
loo = LeaveOneOut()
model = LinearRegression()

scores = cross_val_score(model, X, y, cv=loo, scoring='r2')

print(f"R² promedio: {scores.mean():.3f}")
print(f"R² por observación: min={scores.min():.3f}, max={scores.max():.3f}")
```

---

### ⚖️ Conclusión

**LOOCV** es **teóricamente elegante** pero **prácticamente poco usado**.

🏆 **Recomendación general**: 
- Dataset pequeño (N < 100): **10-Fold CV**
- Dataset mediano/grande: **5-Fold CV**
- LOOCV: Solo si tienes **muy pocos datos** y modelo **muy rápido**

---

## 7️⃣ Validación Espacial con H3

### 🗺️ Problema: Autocorrelación Espacial

**Concepto**: Observaciones **cercanas geográficamente** tienden a ser **similares**.

```
Ejemplo: Ventas de sucursales
├── Sucursal A (zona norte): $10,000/día
├── Sucursal B (zona norte, 1km de A): $9,500/día  ⭐ Similar
└── Sucursal C (zona sur, 20km): $15,000/día        ❌ Diferente
```

❌ **Con K-Fold estándar**:
```
Train: [Zona Norte, Zona Norte, Zona Sur]
Test:  [Zona Norte, Zona Sur]
```

**Problema**: El modelo aprende sobre **Zona Norte** en train y predice **Zona Norte** en test → rendimiento inflado.

✅ **En producción**: Predecirás zonas **nuevas/lejanas** → peor rendimiento real.

---

### ✅ Solución: Spatial Cross-Validation

**Estrategia**: Asegurar que train y test estén **espacialmente separados**.

**Visualización** (hexágonos H3):
```
Fold 1:
  Train: [⬣ ⬣] [ ] [ ] [ ]  (Zona Oeste)
  Test:  [ ] [ ] [⬢ ⬢ ⬢]  (Zona Este)
  
Fold 2:
  Train: [ ] [ ] [⬣ ⬣ ⬣]  (Zona Este)
  Test:  [⬢ ⬢] [ ] [ ] [ ]  (Zona Oeste)
```

✅ **Ventaja**: Evalúa la capacidad del modelo de **generalizar a nuevas zonas geográficas**.

---

### 💻 Implementación con H3

#### Opción 1: Group K-Fold por Zona H3

```python
import h3
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Agrupar por hexágono H3 (resolución 7)
df['h3_zona'] = df.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], 7), axis=1)

X = df[['dia_semana', 'mes', 'temp_celsius']]
y = df['ventas']
groups = df['h3_zona']  # Agrupar por zona

# Group K-Fold espacial
gkf = GroupKFold(n_splits=5)
model = RandomForestRegressor(n_estimators=100, random_state=42)

scores = cross_val_score(model, X, y, groups=groups, cv=gkf, scoring='neg_mean_absolute_error')

print(f"MAE (validación espacial): ${-scores.mean():.2f}")
```

#### Opción 2: Split por Distancia (Custom)

```python
import numpy as np
from sklearn.model_selection import BaseCrossValidator

class SpatialKFold(BaseCrossValidator):
    """Cross-validation espacial custom."""
    
    def __init__(self, n_splits=5, buffer_distance=3):
        self.n_splits = n_splits
        self.buffer_distance = buffer_distance  # Distancia mínima H3
    
    def split(self, X, y=None, groups=None):
        # groups = lista de h3_index
        unique_zones = np.unique(groups)
        np.random.shuffle(unique_zones)
        
        fold_size = len(unique_zones) // self.n_splits
        
        for i in range(self.n_splits):
            # Test zones
            test_zones = unique_zones[i*fold_size:(i+1)*fold_size]
            
            # Train zones (excluir buffer alrededor de test)
            train_zones = []
            for zone in unique_zones:
                if zone not in test_zones:
                    # Verificar distancia mínima a zonas test
                    min_dist = min([h3.h3_distance(zone, tz) for tz in test_zones])
                    if min_dist >= self.buffer_distance:
                        train_zones.append(zone)
            
            train_idx = np.where(np.isin(groups, train_zones))[0]
            test_idx = np.where(np.isin(groups, test_zones))[0]
            
            yield train_idx, test_idx
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

# Usar spatial CV custom
spatial_cv = SpatialKFold(n_splits=5, buffer_distance=3)
scores = cross_val_score(model, X, y, groups=df['h3_index'], cv=spatial_cv, scoring='neg_mean_absolute_error')

print(f"MAE (con buffer espacial): ${-scores.mean():.2f}")
```

---

### 📊 Ejemplo: Predicción de Ventas por Sucursal

**Dataset**: Ventas de 500 clientes en 3 sucursales (zonas H3 diferentes)

```python
import pandas as pd
import h3
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Cargar datos
df_ventas = pd.read_csv('ventas.csv')
df_clientes = pd.read_csv('clientes.csv')

# Merge para obtener H3
df = df_ventas.merge(df_clientes[['cliente_id', 'h3_index']], on='cliente_id')

# Features
X = df[['dia_semana', 'mes', 'segmento_encoded']]
y = df['total']

# Validación espacial
groups = df['h3_index']
gkf = GroupKFold(n_splits=5)
model = RandomForestRegressor(n_estimators=100, random_state=42)

scores_spatial = cross_val_score(model, X, y, groups=groups, cv=gkf, scoring='neg_mean_absolute_error')
scores_standard = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')

print(f"MAE (CV estándar):  ${-scores_standard.mean():.2f}  ❌ Optimista")
print(f"MAE (CV espacial):  ${-scores_spatial.mean():.2f}  ✅ Realista")
print(f"Diferencia:         ${(-scores_spatial.mean()) - (-scores_standard.mean()):.2f}")
```

**Salida esperada**:
```
MAE (CV estándar):  $15.20  ❌ Optimista
MAE (CV espacial):  $18.90  ✅ Realista
Diferencia:         $3.70
```

💡 **La validación espacial revela el verdadero rendimiento en zonas nuevas.**

---

### 🎯 Cuándo Usar Validación Espacial

✅ **Usar cuando**:
- Predicciones **geográficamente distribuidas**
- Datos con **autocorrelación espacial**
- Planeas **expandir a nuevas zonas**
- Tienes features H3 o coordenadas

**Ejemplos**:
- Predicción de precios inmobiliarios
- Forecasting de ventas por sucursal
- Modelos de transporte/logística
- Análisis de sensores distribuidos

---

### ⚠️ Consideraciones

1. **Resolución H3**: Ajustar según escala del problema
   - Resolución 5: ~250 km² (ciudad)
   - Resolución 7: ~5 km² (barrio)
   - Resolución 9: ~0.1 km² (cuadra)

2. **Buffer de distancia**: Evitar zonas "frontera" entre train/test

3. **Distribución desigual**: Algunas zonas tienen más datos que otras

---

# 🎯 Validación Cruzada Avanzada
## Material Complementario - Laboratorio (Herramientas)
### Universidad del Aconcagua - Mendoza, Argentina

---

### 🎯 Objetivos de Aprendizaje

1. Comprender **limitaciones de validación simple** (train/test split)
2. Dominar **K-Fold Cross-Validation** y sus variantes
3. Aplicar **Stratified K-Fold** para datos desbalanceados
4. Usar **Time Series CV** para datos temporales
5. Implementar **Group K-Fold** para evitar data leakage
6. Explorar **validación espacial** con features H3
7. Seleccionar estrategia correcta según tipo de datos

### 📁 Contenido

1. ¿Por qué Cross-Validation?
2. K-Fold Cross-Validation (Revisión)
3. Stratified K-Fold (Datos Desbalanceados)
4. Time Series Cross-Validation
5. Group K-Fold (Evitar Data Leakage)
6. Leave-One-Out CV (LOOCV)
7. Validación Espacial con H3
8. Comparación y Selección de Estrategia
9. Mejores Prácticas

### ⏱️ Duración Estimada: 2-3 horas

---

## 1️⃣ ¿Por qué Necesitamos Cross-Validation?

### 🚨 Problema con Train/Test Split Simple

**Escenario típico**:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
```

❌ **Problemas**:

1. **Varianza alta**: Resultado depende de **cómo se dividieron los datos**
   - Split 1: Accuracy = 85%
   - Split 2: Accuracy = 78%
   - Split 3: Accuracy = 91%
   - **¿Cuál es el verdadero rendimiento?** 🤔

2. **Desperdicio de datos**: Solo 80% se usa para entrenamiento

3. **Sobreajuste al test set**: Si evaluamos múltiples modelos con el mismo test set

---

### ✅ Solución: Cross-Validation

**Concepto**: Dividir datos en **K folds** (particiones) y entrenar K veces, usando cada fold como test una vez.

**Ventajas**:
- ✅ **Estimación más robusta** del rendimiento
- ✅ **Usa todos los datos** para entrenamiento y evaluación
- ✅ **Reduce varianza** de la métrica
- ✅ **Detecta overfitting** más confiablemente

**Ejemplo visual (5-Fold CV)**:
```
Fold 1: [TEST] [TRAIN] [TRAIN] [TRAIN] [TRAIN]  → Score 1
Fold 2: [TRAIN] [TEST] [TRAIN] [TRAIN] [TRAIN]  → Score 2
Fold 3: [TRAIN] [TRAIN] [TEST] [TRAIN] [TRAIN]  → Score 3
Fold 4: [TRAIN] [TRAIN] [TRAIN] [TEST] [TRAIN]  → Score 4
Fold 5: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]  → Score 5

Score final = Mean(Score 1, ..., Score 5) ± Std
```

---

### 📊 ¿Cuándo Usar Cada Tipo?

| Tipo de Datos | Estrategia CV Recomendada |
|---------------|---------------------------|
| 🎲 Datos aleatorios (i.i.d.) | K-Fold estándar |
| ⚖️ Datos desbalanceados | **Stratified K-Fold** |
| 📅 Series temporales | **Time Series CV** |
| 👥 Datos agrupados (clientes, usuarios) | **Group K-Fold** |
| 🗺️ Datos geoespaciales | **Spatial CV** (custom) |
| 🔬 Datasets muy pequeños | **Leave-One-Out CV** |

---

## 2️⃣ K-Fold Cross-Validation (Revisión)

### 📖 Concepto

**K-Fold CV** divide el dataset en **K particiones (folds)** de tamaño similar y entrena K modelos.

**Proceso**:
1. Dividir datos en K folds (ej: K=5)
2. Para cada fold i:
   - Usar fold i como **test set**
   - Usar folds restantes como **training set**
   - Entrenar modelo y calcular métrica
3. Promediar las K métricas

---

### 💻 Implementación Básica

```python
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor

# Crear estrategia K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    cv=kf,  # Estrategia de validación
    scoring='neg_mean_absolute_error',  # Métrica
    n_jobs=-1  # Paralelizar
)

# Resultados
print(f"MAE por fold: {-scores}")
print(f"MAE promedio: {-scores.mean():.2f} ± {scores.std():.2f}")
```

---

### ⚙️ Parámetros Importantes

**`n_splits`**: Número de folds (típicamente 5 o 10)
- Más folds = más tiempo de entrenamiento
- Menos folds = más varianza en estimación
- **Recomendado**: 5 o 10 folds

**`shuffle`**: Mezclar datos antes de dividir
- `True`: Recomendado para datos i.i.d.
- `False`: Para series temporales (mantener orden)

**`random_state`**: Semilla para reproducibilidad

---

### 📊 ¿Cuántos Folds Usar?

| K | Train Size | Test Size | Tiempo | Varianza | Uso |
|---|------------|-----------|--------|----------|-----|
| 3 | 67% | 33% | Rápido | Alta | Experimentación |
| 5 | 80% | 20% | Medio | Media | **Estándar** |
| 10 | 90% | 10% | Lento | Baja | Evaluación final |
| N | N-1 | 1 | Muy lento | Muy baja | Datasets pequeños |

⚖️ **Trade-off**: Más folds = mejor estimación pero más tiempo

---

### ⚠️ Limitaciones de K-Fold Estándar

❌ **No funciona bien con**:
1. **Datos desbalanceados**: Algunos folds pueden no tener clases minoritarias
2. **Series temporales**: Rompe dependencia temporal (entrena con futuro)
3. **Datos agrupados**: Puede haber data leakage (mismo cliente en train y test)
4. **Datos espaciales**: No considera autocorrelación espacial

✅ **Solución**: Usar variantes especializadas (próximas secciones)

---

## 3️⃣ Stratified K-Fold (Datos Desbalanceados)

### 🎯 Problema que Resuelve

**Escenario**: Dataset de clasificación con clases **desbalanceadas**.

```
Dataset: 1000 registros
├── Clase A: 900 registros (90%)
└── Clase B: 100 registros (10%)
```

❌ **Con K-Fold estándar**:
- Algunos folds pueden tener **0% de clase B**
- Modelo no puede aprender clase minoritaria en esos folds
- Métricas sesgadas

✅ **Con Stratified K-Fold**:
- **Mantiene la proporción** de clases en cada fold
- Cada fold tiene ~90% clase A y ~10% clase B
- Evaluación más justa

---

### 📊 Visualización

**K-Fold estándar** (distribución aleatoria):
```
Fold 1: A=950, B=50   (95% A, 5% B)   ❌ Desbalanceado
Fold 2: A=850, B=150  (85% A, 15% B)  ❌ Desbalanceado
Fold 3: A=920, B=80   (92% A, 8% B)   ❌ Desbalanceado
```

**Stratified K-Fold** (proporción preservada):
```
Fold 1: A=900, B=100  (90% A, 10% B)  ✅ Balanceado
Fold 2: A=900, B=100  (90% A, 10% B)  ✅ Balanceado
Fold 3: A=900, B=100  (90% A, 10% B)  ✅ Balanceado
```

---

### 💻 Implementación

#### Clasificación (Estándar)
```python
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Crear estrategia Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Cross-validation
scores = cross_val_score(
    model, X, y, 
    cv=skf,  # Estrategia stratified
    scoring='f1_weighted'
)

print(f"F1-Score: {scores.mean():.3f} ± {scores.std():.3f}")
```

#### Regresión (Binning Manual)
```python
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Para regresión, crear bins del target
y_binned = pd.qcut(y, q=5, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Iterar manualmente
for train_idx, test_idx in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    print(f"Score: {score:.3f}")
```

---

### 📈 Caso de Uso: Segmentación de Clientes

**Dataset de Panadería**:
- Segmento Premium: 5% de clientes
- Segmento Regular: 70% de clientes
- Segmento Ocasional: 25% de clientes

✅ **Stratified K-Fold** asegura que cada fold tenga la misma distribución.

---

### ⚙️ Parámetros Importantes

**Igual que K-Fold**, pero con estratificación automática:
- `n_splits`: Número de folds
- `shuffle`: Mezclar datos (recomendado `True`)
- `random_state`: Reproducibilidad

---

### 🎯 Cuándo Usar Stratified K-Fold

✅ **Usar cuando**:
- Clasificación con **clases desbalanceadas**
- Regresión con **distribución sesgada** del target
- Quieres **garantizar representatividad** en cada fold

❌ **No usar cuando**:
- Datos temporales (usar Time Series CV)
- Datos agrupados (usar Group K-Fold)

---

## 4️⃣ Time Series Cross-Validation

### 🚨 Problema con CV Estándar en Series Temporales

❌ **K-Fold estándar ROMPE la dependencia temporal**:

```
Datos: [Ene, Feb, Mar, Abr, May, Jun, Jul, Ago, Sep]

K-Fold estándar (INCORRECTO):
Fold 1: Train=[Feb,Mar,May,Jun,Ago,Sep]  Test=[Ene,Abr,Jul]  ⛔ Entrena con futuro!
Fold 2: Train=[Ene,Mar,Abr,Jun,Jul,Sep]  Test=[Feb,May,Ago]  ⛔ Entrena con futuro!
```

**Problema**: El modelo **ve el futuro** durante entrenamiento → **data leakage**

---

### ✅ Solución: Time Series Split (Forward Chaining)

**Concepto**: Entrenar solo con **datos pasados**, nunca con datos futuros.

**Visualización**:
```
Fold 1: [TRAIN] [TEST] [ ] [ ] [ ]      → Predice paso 2
Fold 2: [TRAIN] [TRAIN] [TEST] [ ] [ ]  → Predice paso 3
Fold 3: [TRAIN] [TRAIN] [TRAIN] [TEST] [ ] → Predice paso 4
Fold 4: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST] → Predice paso 5
```

**Características**:
- ✅ Respeta **orden temporal**
- ✅ Training set **crece** con cada fold
- ✅ Test set siempre **después** de train
- ✅ Simula **producción real** (predecir futuro)

---

### 💻 Implementación con TimeSeriesSplit

```python
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Time Series CV con 5 splits
tscv = TimeSeriesSplit(n_splits=5)

model = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation
scores = cross_val_score(
    model, X, y, 
    cv=tscv,  # Time Series strategy
    scoring='neg_mean_absolute_error'
)

print(f"MAE por fold: {-scores}")
print(f"MAE promedio: {-scores.mean():.2f} ± {scores.std():.2f}")
```

---

### ⚙️ Variantes de Time Series CV

#### 1. **Expanding Window** (Ventana Creciente) - Default

```
Fold 1: [█] [T]
Fold 2: [█ █] [T]
Fold 3: [█ █ █] [T]
Fold 4: [█ █ █ █] [T]
```

✅ Usa **todos los datos históricos**  
⚠️ Puede ser lento con datasets grandes

#### 2. **Rolling Window** (Ventana Deslizante)

```
Fold 1: [█ █] [T] [ ]
Fold 2: [ ] [█ █] [T]
Fold 3: [ ] [ ] [█ █] [T]
```

✅ **Tamaño fijo** de entrenamiento  
✅ Más rápido  
⚠️ Descarta datos antiguos

**Implementación manual**:
```python
from sklearn.model_selection import TimeSeriesSplit

# Rolling window custom
window_size = 1000  # Últimos 1000 registros

for train_idx, test_idx in TimeSeriesSplit(n_splits=5).split(X):
    # Limitar train a últimos `window_size` registros
    train_idx = train_idx[-window_size:]
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
```

---

### 📊 Ejemplo: Ventas de Panadería por Día

**Dataset**: Ventas diarias de enero a diciembre (365 días)

```python
# Ordenar por fecha (IMPORTANTE)
df = df.sort_values('fecha').reset_index(drop=True)

# Preparar features y target
X = df[['dia_semana', 'mes', 'es_feriado', 'temp_celsius']]
y = df['ventas_totales']

# Time Series CV
tscv = TimeSeriesSplit(n_splits=6)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    print(f"Fold {fold}:")
    print(f"  Train: días {train_idx[0]} a {train_idx[-1]}")
    print(f"  Test:  días {test_idx[0]} a {test_idx[-1]}")
```

**Output esperado**:
```
Fold 1:
  Train: días 0 a 59    (Ene-Feb)
  Test:  días 60 a 119  (Mar-Abr)
  
Fold 2:
  Train: días 0 a 119   (Ene-Abr)
  Test:  días 120 a 179 (May-Jun)
...
```

---

### 🎯 Cuándo Usar Time Series CV

✅ **Siempre que tengas dependencia temporal**:
- Predicción de ventas
- Forecasting de demanda
- Predicción de precios
- Series temporales en general

❌ **No usar cuando**:
- Datos NO tienen orden temporal
- Quieres aprovechar datos futuros (no es realista)

---

### ⚠️ Errores Comunes

1. **No ordenar datos por fecha** antes de split
2. **Usar K-Fold estándar** en series temporales
3. **Incluir features con "fuga del futuro"** (ej: precio_futuro)
4. **No considerar estacionalidad** en los folds

---